# Order Est Income for Landlord

Hitung estimated income **per order** (1 baris per `order_id + transaction_id`, split payment = 2 baris) dari `transactions_iseller_pusat` (data lake), terus dump ke `order_est_income_for_landlord` (DWH). Dipakai buat revenue sharing landlord outlet 00570. Hasilnya divalidasi dulu vs `daily_est_income_per_store_for_landlord` sebelum di-dump.

Created at : 2026-11-20

Created by lahiardhan@jiwagroup.com

# IMPORT LIBRARY

In [1]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import psycopg2

import requests
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import pymysql
pd.set_option('display.max_columns', None)

# CONFIG DB

In [2]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "../db_config/config_db_datawarehouse.py")

In [3]:
def dl_dwh_db_conn(db_name):
    return pymysql.connect(
        host=config_dl_dwh.hostname,
        db=db_name,
        user=config_dl_dwh.username,
        password=config_dl_dwh.password,
        port=3306
    )

# RULES / CONFIG PERHITUNGAN

Semua rules besaran komisi per payment type ditaruh di sini (bukan di query), biar gampang diubah tanpa nyentuh SQL. `PAYMENT_TYPE_MAP` & `COMMISSION_PCT` harus identik dengan notebook daily, biar hasilnya sama.

Table target level order: satu baris per `order_id + transaction_id` (split payment = 2 baris).

In [4]:
# outlet yang mau dihitung (dipakai di query extract & DELETE dump)
OUTLET_CODES = ['00570']   # tambah outlet baru cukup di sini, mis. ['00570', '01165']

# mapping raw transactions_payment_type_name (dari DB) -> payment type resmi
# (label resmi ini juga yang dikirim ke landlord & dicatat di api_landlord_log_detail.payment_type)
PAYMENT_TYPE_MAP = {
    'Cash': 'Cash',
    'BRI': 'BRI',
    'Go Food': 'Go Food',
    'Grab Food': 'Grab Food',
    'Grab Dine Out': 'Grab Dine Out',
    'Shopee Food': 'Shopee Food',
    'Tokopedia Go': 'Tokopedia Go',
    'Jiwa+': 'Jiwa+',
    'iSellerpay (Sejutajiwariders)': 'iSeller QRIS',
    'iSeller Pay QRIS': 'iSeller QRIS',
    'isellerpay_qris': 'iSeller QRIS',
    'Mandiri QRIS': 'Mandiri QRIS',
    'QPON': 'QPON',
}
UNKNOWN_PAYMENT_TYPE = 'Unknown'  # raw payment_type NULL

# besaran commission cost per payment type resmi (pct dari subtotal / net sales)
COMMISSION_PCT = {
    'Cash': 0.0,
    'BRI': 0.0,          # all banks
    'Go Food': 0.16,
    'Grab Food': 0.15,
    'Grab Dine Out': 0.15,
    'Shopee Food': 0.15,
    'Tokopedia Go': 0.07,
    'Jiwa+': 0.0,
    'iSeller QRIS': 0.007,
    'Mandiri QRIS': 0.0,
    'QPON': 0.05,
}

# kolom final sesuai urutan di table (di luar load_data_at & load_data_by)
ORDER_COLUMNS = ['date', 'transaction_date', 'outlet_code', 'outlet_name', 'order_id', 'transaction_id',
                 'order_reference', 'channel', 'payment_type_raw', 'payment_type',
                 'total_order_amount', 'total_discount_amount', 'total_promotion_amount',
                 'subtotal', 'tax_amount', 'total_sales', 'commission_pct', 'sales_type_commission']

# FUNCTION

In [5]:
def get_from_dl_dwh():
    connection = dl_dwh_db_conn(config_dl_dwh.db_resource)
    cursor = connection.cursor()

    outlet_list = "', '".join(OUTLET_CODES)

    sql = f"""
        SELECT
            DATE_FORMAT(transaction_date, '%Y-%m-%d') AS date,
            transaction_date,
            outlet_code,
            outlet_name,
            order_id,
            transaction_id,
            order_reference,
            channel,
            transactions_payment_type_name AS payment_type_raw,
            total_order_amount,
            total_discount_amount,
            total_promotion_amount,
            subtotal,
            total_tax_amount AS tax_amount,
            total_amount AS total_sales
        FROM transactions_iseller_pusat
        WHERE DATE_FORMAT(transaction_date, '%Y-%m-%d') >= '{start_date_str}'
            AND DATE_FORMAT(transaction_date, '%Y-%m-%d') <= '{end_date_str}'
            AND outlet_code IN ('{outlet_list}')
            AND fulfillment_status = 'fulfilled'
            AND status = 'completed'
            AND payment_status = 'paid'
            AND transactions_status = 'success'
        ORDER BY transaction_date
    """

    cursor.execute(sql)

    record = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()

    return df

In [6]:
# lookup tidak sensitif huruf besar/kecil & spasi di pinggir (kolom di MySQL juga case-insensitive)
_NORMALIZED_MAP = {k.strip().casefold(): v for k, v in PAYMENT_TYPE_MAP.items()}

def map_payment_type(raw_payment_type):
    if pd.isna(raw_payment_type):
        return UNKNOWN_PAYMENT_TYPE
    raw = str(raw_payment_type).strip()
    return _NORMALIZED_MAP.get(raw.casefold(), raw)

# CONFIG DATE RANGE

Default harian: jalan tiap hari proses H-1 (`yesterday`) dengan `USE_CUSTOM_DATE_RANGE = False`. Kalau mau custom range (backfill/reprocess/verifikasi), set `USE_CUSTOM_DATE_RANGE = True` lalu isi `start_date` & `end_date`.

In [7]:
USE_CUSTOM_DATE_RANGE = True  # set True kalau mau pakai start_date & end_date custom di bawah (run harian pakai False -> H-1)

yesterday = date.today() - timedelta(days=1)

# tanggal verifikasi (sudah ada di log API)
start_date = date(2026, 1, 2)
end_date   = date(2026, 1, 5)

if USE_CUSTOM_DATE_RANGE:
    run_start_date = start_date
    run_end_date = end_date
else:
    run_start_date = yesterday
    run_end_date = yesterday

start_date_str = run_start_date.strftime('%Y-%m-%d')
end_date_str   = run_end_date.strftime('%Y-%m-%d')

print("start_date:", start_date_str)
print("end_date  :", end_date_str)

start_date: 2026-01-02
end_date  : 2026-01-05


# MAIN PROGRESS

In [8]:
df_raw = get_from_dl_dwh()
print("jumlah baris :", len(df_raw))
print("jumlah order :", df_raw['order_id'].nunique())
df_raw.head()

jumlah baris : 129
jumlah order : 129


,date,transaction_date,outlet_code,outlet_name,order_id,transaction_id,order_reference,channel,payment_type_raw,total_order_amount,total_discount_amount,total_promotion_amount,subtotal,tax_amount,total_sales
0,2026-01-02,2026-01-02 08:00:52,00570,00570 DISTRICT 2 MEIKARTA CKR,3861848b-28c4-4bef-bd2f-478a772e00d4,8fd17ddd-37de-4876-b452-d18273518d2b,GF/GF-580/PHPK.020126,web,Grab Food,56789.516790,9588.181818,18772.727273,28428.607699,2874.0000,31303.0
1,2026-01-02,2026-01-02 08:55:44,00570,00570 DISTRICT 2 MEIKARTA CKR,be34ae63-acf6-4bd1-b72e-2516a42c79bc,72f92b27-526c-4396-8545-29c09298abbb,#2630-6177,pos,iSeller Pay QRIS,16363.636364,0.000000,0.000000,16363.636364,1636.3636,18000.0
2,2026-01-02,2026-01-02 09:10:33,00570,00570 DISTRICT 2 MEIKARTA CKR,267d0ea2-ed91-4e31-8621-c6fb5f812b74,87879726-5bd5-4d25-b056-6ea628271c51,GF/GF-196/9I9E.020126,web,Grab Food,53153.153153,0.000000,2727.272727,50425.880426,5074.0000,55500.0
3,2026-01-02,2026-01-02 09:20:33,00570,00570 DISTRICT 2 MEIKARTA CKR,0f8d22b9-d31e-4d30-8f13-1bec80c02db5,0a204e54-40ad-4094-bfd6-04e5afa608e1,TA/BRYAN/SQ2W.020126,web,Jiwa+,40909.090909,0.000000,22727.272727,18181.818182,1818.0000,20000.0
4,2026-01-02,2026-01-02 09:58:16,00570,00570 DISTRICT 2 MEIKARTA CKR,652557bd-1127-4ff4-ba20-ec70b8967bc1,fd7eaa09-01c4-4e0f-a578-df662ea5eff5,#2630-6178,pos,iSeller Pay QRIS,20000.000000,0.000000,0.000000,20000.000000,1999.9999,22000.0


In [9]:
df = df_raw.copy()

# MySQL balikin Decimal -> float supaya bisa dikali pct
for col in ['total_order_amount', 'total_discount_amount', 'total_promotion_amount',
            'subtotal', 'tax_amount', 'total_sales']:
    df[col] = pd.to_numeric(df[col]).astype(float)

df['payment_type'] = df['payment_type_raw'].apply(map_payment_type)

# komisi (rupiah) per baris, tidak dibulatkan (kolom DECIMAL(20,4) yang membatasi 4 desimal); uang masuk landlord sengaja gak disimpan -> subtotal - sales_type_commission
df['commission_pct'] = df['payment_type'].map(COMMISSION_PCT).fillna(0.0)
df['sales_type_commission'] = df['subtotal'] * df['commission_pct']

# payment type yang belum ada di COMMISSION_PCT: tetap disimpan (komisi 0), tapi cek dulu apakah perlu didaftarkan
unmapped = df[~df['payment_type'].isin(COMMISSION_PCT)]
if unmapped.empty:
    print("Semua payment type sudah dikenal.")
else:
    print("WARNING payment type belum dikenal (komisi dianggap 0):")
    print(unmapped.groupby('payment_type')['total_sales'].sum())

# fail loud: duplikat order_id + transaction_id (PK table juga bakal gagal)
dup = df[df.duplicated(['order_id', 'transaction_id'], keep=False)]
if not dup.empty:
    print("DUPLIKAT order_id + transaction_id:")
    print(dup.sort_values(['order_id', 'transaction_id']))
    raise ValueError(f"Ada {len(dup)} baris duplikat (order_id, transaction_id), cek sumber dulu.")

# split payment = order_id dengan >1 baris
split = df[df.duplicated('order_id', keep=False)]
if split.empty:
    print("\ntidak ada split payment")
else:
    print(f"\nsplit payment: {split['order_id'].nunique()} order")
    print(split.sort_values(['order_id', 'transaction_date'])[['date', 'order_id', 'order_reference', 'transaction_id', 'payment_type', 'subtotal']])

df = df[ORDER_COLUMNS]
df.head(10)

Semua payment type sudah dikenal.

tidak ada split payment


,date,transaction_date,outlet_code,outlet_name,order_id,transaction_id,order_reference,channel,payment_type_raw,payment_type,total_order_amount,total_discount_amount,total_promotion_amount,subtotal,tax_amount,total_sales,commission_pct,sales_type_commission
0,2026-01-02,2026-01-02 08:00:52,00570,00570 DISTRICT 2 MEIKARTA CKR,3861848b-28c4-4bef-bd2f-478a772e00d4,8fd17ddd-37de-4876-b452-d18273518d2b,GF/GF-580/PHPK.020126,web,Grab Food,Grab Food,56789.516790,9588.181818,18772.727273,28428.607699,2874.0000,31303.0,0.150,4264.291155
1,2026-01-02,2026-01-02 08:55:44,00570,00570 DISTRICT 2 MEIKARTA CKR,be34ae63-acf6-4bd1-b72e-2516a42c79bc,72f92b27-526c-4396-8545-29c09298abbb,#2630-6177,pos,iSeller Pay QRIS,iSeller QRIS,16363.636364,0.000000,0.000000,16363.636364,1636.3636,18000.0,0.007,114.545455
2,2026-01-02,2026-01-02 09:10:33,00570,00570 DISTRICT 2 MEIKARTA CKR,267d0ea2-ed91-4e31-8621-c6fb5f812b74,87879726-5bd5-4d25-b056-6ea628271c51,GF/GF-196/9I9E.020126,web,Grab Food,Grab Food,53153.153153,0.000000,2727.272727,50425.880426,5074.0000,55500.0,0.150,7563.882064
3,2026-01-02,2026-01-02 09:20:33,00570,00570 DISTRICT 2 MEIKARTA CKR,0f8d22b9-d31e-4d30-8f13-1bec80c02db5,0a204e54-40ad-4094-bfd6-04e5afa608e1,TA/BRYAN/SQ2W.020126,web,Jiwa+,Jiwa+,40909.090909,0.000000,22727.272727,18181.818182,1818.0000,20000.0,0.000,0.000000
4,2026-01-02,2026-01-02 09:58:16,00570,00570 DISTRICT 2 MEIKARTA CKR,652557bd-1127-4ff4-ba20-ec70b8967bc1,fd7eaa09-01c4-4e0f-a578-df662ea5eff5,#2630-6178,pos,iSeller Pay QRIS,iSeller QRIS,20000.000000,0.000000,0.000000,20000.000000,1999.9999,22000.0,0.007,140.000000
5,2026-01-02,2026-01-02 10:44:10,00570,00570 DISTRICT 2 MEIKARTA CKR,f1b11308-55b1-4ff5-83fa-d74009fd0fc7,23d3f46b-fc86-4108-88a6-e9ce6a003411,#2630-6179,pos,iSeller Pay QRIS,iSeller QRIS,45454.545455,0.000000,0.000000,45454.545455,4545.4545,50000.0,0.007,318.181818
6,2026-01-02,2026-01-02 10:52:31,00570,00570 DISTRICT 2 MEIKARTA CKR,d7ea0d56-9331-4ddb-8114-d2d949fbfc76,f05b03d8-5fbc-4cfb-92e7-89683d6eecd8,TA/ANISSA/9WVU.020126,web,Jiwa+,Jiwa+,45454.545455,1060.000000,22727.272727,21667.272727,2166.0000,23834.0,0.000,0.000000
7,2026-01-02,2026-01-02 10:53:47,00570,00570 DISTRICT 2 MEIKARTA CKR,19edb344-ffa0-4989-8d72-d263c35c17b5,078b8527-b776-468d-b53a-9987801189b4,DEL/WIDYA/MUC6.020126,web,Jiwa+,Jiwa+,179033.579034,0.000000,81818.181818,97215.397215,9784.0000,107000.0,0.000,0.000000
8,2026-01-02,2026-01-02 11:15:16,00570,00570 DISTRICT 2 MEIKARTA CKR,03649e3e-4e6c-41c7-90e3-4e580a19fa09,3a0ba7eb-9ce6-4c39-b3d1-a9a22aa288c0,TA/UNAA/X861.020126,web,Jiwa+,Jiwa+,20000.000000,2000.000000,0.000000,18000.000000,1800.0000,19800.0,0.000,0.000000
9,2026-01-02,2026-01-02 11:35:17,00570,00570 DISTRICT 2 MEIKARTA CKR,21169bfe-c306-4655-a466-f0e35380bbab,4ccc3089-55a4-4de1-8e79-51d0e95f23b5,GF/GF-213/6XXG.020126,web,Grab Food,Grab Food,46789.516790,10000.000000,2727.272727,34062.244062,3438.0000,37500.0,0.150,5109.336609


# VALIDASI VS DAILY

Agregat level order dibandingkan dengan `daily_est_income_per_store_for_landlord` di range & outlet yang sama. Kalau ada selisih, `raise` supaya DUMP tidak jalan.

In [10]:
# ambil tabel daily di range & outlet yang sama
connection = dl_dwh_db_conn(config_dl_dwh.db_target)
cursor = connection.cursor()
outlet_list = "', '".join(OUTLET_CODES)
cursor.execute(f"""
    SELECT date, outlet_code, payment_type, subtotal, tax_amount, total_sales, sales_type_commission, total_order
    FROM daily_est_income_per_store_for_landlord
    WHERE date BETWEEN '{start_date_str}' AND '{end_date_str}'
        AND outlet_code IN ('{outlet_list}')
""")
daily = pd.DataFrame(cursor.fetchall(), columns=[d[0] for d in cursor.description])
cursor.close()
connection.close()

money_cols = ['subtotal', 'tax_amount', 'total_sales', 'sales_type_commission']
for col in money_cols + ['total_order']:
    daily[col] = pd.to_numeric(daily[col]).astype(float)
daily['date'] = daily['date'].astype(str)
daily['_pt_key'] = daily['payment_type'].str.casefold()

# agregat order-level, kunci payment type tidak sensitif huruf besar/kecil (sama kayak daily)
chk = df.copy()
chk['_pt_key'] = chk['payment_type'].str.casefold()
agg = (
    chk.groupby(['date', 'outlet_code', '_pt_key'], as_index=False)
        .agg(payment_type=('payment_type', 'first'),
             subtotal=('subtotal', 'sum'),
             tax_amount=('tax_amount', 'sum'),
             total_sales=('total_sales', 'sum'),
             sales_type_commission=('sales_type_commission', 'sum'),
             total_order=('order_id', 'nunique'))
)

cmp = agg.merge(daily, on=['date', 'outlet_code', '_pt_key'], how='outer', suffixes=('_order', '_daily'), indicator=True)
for col in money_cols + ['total_order']:
    cmp[f'diff_{col}'] = cmp[f'{col}_order'].fillna(0) - cmp[f'{col}_daily'].fillna(0)

diff_cols = [f'diff_{c}' for c in money_cols + ['total_order']]
display(cmp[['date', 'outlet_code', '_pt_key', '_merge'] + diff_cols].sort_values(['date', 'outlet_code', '_pt_key']))

only_one_side = cmp[cmp['_merge'] != 'both']
bad_money = (cmp[[f'diff_{c}' for c in money_cols]].abs() >= 0.01).any(axis=1)
bad_order = cmp['diff_total_order'] != 0
if not only_one_side.empty or bad_money.any() or bad_order.any():
    print("ada baris yang beda / hanya di salah satu sisi:")
    print(cmp[(cmp['_merge'] != 'both') | bad_money | bad_order])
    raise ValueError("Validasi vs daily GAGAL, jangan dump. Cek tabel diff di atas.")
print("Validasi vs daily OK: semua diff < 0.01 dan total_order sama.")

# referensi: uang masuk per tanggal (subtotal - komisi) buat dibandingin dgn yg dikirim ke API
# (00570, 2026-10-14: total 2,140,812.0314; Cash 87,272.7273)
df['uang_masuk'] = df['subtotal'] - df['sales_type_commission']
print("\nuang masuk per tanggal:")
print(df.groupby('date')['uang_masuk'].sum().round(4))
print("\nuang masuk per tanggal & payment type:")
print(df.groupby(['date', 'payment_type'])['uang_masuk'].sum().round(4))
df = df.drop(columns='uang_masuk')

,date,outlet_code,_pt_key,_merge,diff_subtotal,diff_tax_amount,diff_total_sales,diff_sales_type_commission,diff_total_order
0,2026-01-02,00570,bri,both,-3.636364e-05,0.000000e+00,0.0,0.000000e+00,0.0
1,2026-01-02,00570,go food,both,2.129440e-06,0.000000e+00,0.0,4.340709e-06,0.0
2,2026-01-02,00570,grab food,both,-4.610972e-05,0.000000e+00,0.0,3.808354e-05,0.0
3,2026-01-02,00570,iseller qris,both,4.545471e-05,-7.275958e-12,0.0,-1.818182e-05,0.0
4,2026-01-02,00570,jiwa+,both,4.267000e-05,0.000000e+00,0.0,0.000000e+00,0.0
5,2026-01-02,00570,qpon,both,-3.636363e-05,0.000000e+00,0.0,1.818182e-05,0.0
6,2026-01-02,00570,shopee food,both,2.942752e-05,0.000000e+00,0.0,-4.558588e-05,0.0
7,2026-01-03,00570,bri,both,-1.957407e-05,0.000000e+00,0.0,0.000000e+00,0.0
8,2026-01-03,00570,go food,both,-1.957412e-05,0.000000e+00,0.0,4.086814e-05,0.0
9,2026-01-03,00570,grab food,both,3.357904e-05,0.000000e+00,0.0,-4.496314e-05,0.0


Validasi vs daily OK: semua diff < 0.01 dan total_order sama.

uang masuk per tanggal:
date
2026-01-02    1.757501e+06
2026-01-03    2.376669e+06
2026-01-04    9.115926e+05
2026-01-05    1.025696e+06
Name: uang_masuk, dtype: float64

uang masuk per tanggal & payment type:
date        payment_type
2026-01-02  BRI              16363.6364
            Go Food         144985.7887
            Grab Food       242236.3067
            Jiwa+           379892.6699
            QPON             34545.4545
            Shopee Food     289513.8123
            iSeller QRIS    649963.6364
2026-01-03  BRI             575880.4259
            Go Food          45595.5577
            Grab Food        58678.5422
            Jiwa+           565314.5455
            QPON            299681.8182
            Shopee Food     196391.0533
            Tokopedia Go     25363.6364
            iSeller QRIS    609763.8084
2026-01-04  Go Food         228170.8156
            Grab Food       146358.7224
            Jiwa+     

# DUMP

In [11]:
from sqlalchemy import text

TARGET_TABLE = 'order_est_income_for_landlord'
LOAD_DATA_BY = 'lahiardhan@jiwagroup.com'   # run manual dari notebook = email operator (NEXUS_AIRFLOW khusus load otomatis)

df_load = df.copy()
df_load['load_data_at'] = datetime.now(timezone('Asia/Jakarta')).replace(tzinfo=None)
df_load['load_data_by'] = LOAD_DATA_BY

engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(hostsname=config_dl_dwh.hostname,
                                                                             user=config_dl_dwh.username,
                                                                             pw=config_dl_dwh.password,
                                                                             db=config_dl_dwh.db_target))

if df_load.empty:
    print("Tidak ada data, table tidak disentuh.")
else:
    # DELETE range lalu INSERT dalam satu transaksi (gagal salah satu -> rollback), aman di-run ulang
    outlet_params = {f'o{i}': o for i, o in enumerate(OUTLET_CODES)}
    outlet_in = ', '.join(f':{k}' for k in outlet_params)
    with engine.begin() as conn:
        deleted = conn.execute(
            text(f"DELETE FROM {TARGET_TABLE} WHERE date BETWEEN :s AND :e AND outlet_code IN ({outlet_in})"),
            {'s': start_date_str, 'e': end_date_str, **outlet_params},
        ).rowcount
        df_load.to_sql(TARGET_TABLE, con=conn, if_exists='append', index=False)
    print(f"{TARGET_TABLE}: -{deleted} / +{len(df_load)} rows ({start_date_str} s/d {end_date_str})")

order_est_income_for_landlord: -0 / +129 rows (2026-01-02 s/d 2026-01-05)
